In [83]:
import pandas as pd

In [84]:
df = pd.read_csv("tests.csv")

In [85]:
import re
from typing import Optional, Union


def parse_response(response: Optional[Union[int, str]]):
    """Parse response containing only a score."""
    if isinstance(response, int):
        return response
    match = re.match(r"^[^\d]*(\d+)", response)
    if match:
        score = int(match.group(1))  # Extract score as an integer
    else:
        print("Could not parse response")
        score = 0
    return score


if not df.responses.dtype == int:
    df.responses = df.responses.map(parse_response)
df.responses

0      3
1      3
2      3
3      3
4      3
      ..
296    4
297    4
298    3
299    3
300    3
Name: responses, Length: 301, dtype: int64

In [86]:
from sklearn.metrics import precision_score, recall_score

records = []
for threshold in range(0, 10):
    prediction = (df.responses > threshold).astype(int)
    label = (df.labels == 10).astype(int)
    records.append(
        {
            "threshold": threshold + 1,
            "recall": recall_score(label, prediction, average=None)[1],
            "precision": precision_score(label, prediction, average=None)[1],
        }
    )
df_metrics = pd.DataFrame.from_records(records)

/Users/giuseppeguarino/Documents/d4g/climateguard/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/giuseppeguarino/Documents/d4g/climateguard/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [87]:
df_metrics

,threshold,recall,precision
0,1,0.955224,0.218430
1,2,0.955224,0.218430
2,3,0.955224,0.219178
3,4,0.626866,0.262500
4,5,0.328358,0.261905
5,6,0.119403,0.275862
6,7,0.000000,0.000000
7,8,0.000000,0.000000
8,9,0.000000,0.000000
9,10,0.000000,0.000000


In [88]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_double_data(x, y1, y2, x_title="", y1_title="", y2_title="", y_range=None):
    # Create figure with secondary y-axis
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Add traces
    fig.add_trace(
        go.Scatter(x=x, y=y1, name=y1_title),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(x=x, y=y2, name=y2_title),
        secondary_y=True,
    )

    # Add figure title
    fig.update_layout(title_text="Evolution of Precision and Recall with Threshold")
    if y_range:
        if not (isinstance(y_range[0], list) or isinstance(y_range[0], tuple)):
            fig.update_layout(yaxis_range=y_range, yaxis2_range=y_range)
        else:
            fig.update_layout(yaxis_range=y_range[0], yaxis2_range=y_range[1])

    # Set x-axis title
    fig.update_xaxes(title_text=f"<b>{x_title.title()}</b>")
    # for trace_data in fig.data:
    #     print(trace_data)

    # Set y-axes titles
    fig.update_yaxes(title_text=f"<b>{y1_title.title()}</b>", secondary_y=False)
    fig.update_yaxes(title_text=f"<b>{y2_title.title()}</b>", secondary_y=True)

    return fig

In [89]:
plot_double_data(
    df_metrics.threshold,
    df_metrics.precision,
    df_metrics.recall,
    "threshold",
    "precision",
    "recall",
    y_range=(-0.1, 1),
)

In [90]:
from sklearn.metrics import precision_score, recall_score

records = []
for threshold in range(0, 10):
    prediction = (df.responses > threshold).astype(int)
    label = (df.labels == 10).astype(int)

    records.append(
        {
            "threshold": threshold + 1,
            "n_predicted_mesinfo": len(prediction[prediction == 1]),
            "n_actual_mesinfo": len(label[label == 1]),
        }
    )
df_counts = pd.DataFrame.from_records(records)

In [91]:
plot_double_data(
    df_counts.threshold,
    df_counts.n_predicted_mesinfo,
    df_counts.n_actual_mesinfo,
    "threshold",
    "num predictedmesinfo",
    "num actual mesinfo",
)